# Day 1 Comparing Models for Tamweel Lite

**Naif Faraj AlDosari · SDA-DSC-211**

I want to see which model ranks applications by default risk better when all three are tested on the same data.

The outcome is `default_within_90d`. The model uses synthetic application data and predictors that are available when the application is submitted. This comparison gives me a candidate for the next lab.

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Day 1 · Build a baseline and compare boosting</h2>
<p><strong>SDA-DSC-211 · Tamweel Lite · 60-minute lab · Free CPU</strong></p>
<p>Train Logistic Regression, XGBoost and LightGBM on the same development and comparison rows. Compare ranking quality and training time, then select an initial candidate using evidence from your own run.</p>
<p><strong>Deliverables:</strong> model-comparison CSV, prediction evidence, split membership, learning curves, ROC/precision–recall figure, run metadata and one short daily answer in README.</p>
<p><strong>Important boundary:</strong> the stratified random split is a temporary teaching baseline. It does not separate repeated customers or reproduce time and target-maturity boundaries. Do not present these scores as a deployment estimate. Day 2 replaces this design with honest validation.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>اليوم الأول · ابنِ خط أساس وقارن نماذج التعزيز</h2>
<p><strong>SDA-DSC-211 · Tamweel Lite · لاب 60 دقيقة · CPU مجاني</strong></p>
<p>درّب Logistic Regression وXGBoost وLightGBM على صفوف التطوير والمقارنة نفسها. قارن جودة ترتيب المخاطر وزمن التدريب، ثم اختر مرشحًا أوليًا اعتمادًا على أدلة تشغيلك أنت.</p>
<p><strong>المخرجات:</strong> ملف مقارنة النماذج، وأدلة التنبؤ، وعضوية التقسيم، ومنحنيات التعلم، ورسم ROC وprecision–recall، وبيانات التشغيل، وإجابة يومية قصيرة في README.</p>
<p><strong>حد مهم:</strong> التقسيم العشوائي الطبقي خط أساس تعليمي مؤقت؛ لا يفصل العملاء المتكررين ولا يحاكي حدود الزمن ونضج الهدف. لا تعرض هذه الدرجات بوصفها تقديرًا للنشر. سيستبدل اليوم الثاني هذا التصميم بتحقق صادق.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Official daily requirement</h2>
<p>Run all cells, compare the generated files with EXPECTED_RESULTS.md, then answer only the single question at the end of this notebook in your repository README. Legacy reflection variables inside code cells are compatibility fields; leave them unchanged.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>المطلوب اليومي المعتمد</h2>
<p>شغّل جميع الخلايا، وقارن الملفات الناتجة بصفحة EXPECTED_RESULTS.md، ثم أجب فقط عن السؤال الواحد في نهاية هذا الدفتر داخل README في مستودعك. حقول الانعكاس القديمة داخل خلايا الكود مخصصة للتوافق؛ اتركها دون تعديل.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>1. Prepare the environment</h2>
<p>Select <strong>Runtime → Run all</strong> using the free CPU runtime. The setup downloads pinned course files, verifies SHA-256 hashes and retries temporary network failures.</p>
<ul><li><code>FAST_MODE=True</code>: up to 300 trees and two CPU threads; use this first.</li><li><code>FULL_MODE=True</code>: optional cap of 900 trees after saving the baseline run.</li><li>No GPU, API key, Drive mount or paid subscription is required.</li></ul>
<p><strong>Pass condition:</strong> the setup prints <code>Day 1 setup verified</code>. A checksum or imported-version failure must not be bypassed; restart the session and run from the beginning.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>1. جهّز بيئة التشغيل</h2>
<p>اختر <strong>Runtime → Run all</strong> باستخدام CPU المجاني. ينزّل الإعداد ملفات الدورة ذات الإصدارات المثبتة، ويتحقق من بصمات SHA-256، ويعيد محاولة أخطاء الشبكة المؤقتة.</p>
<ul><li><code>FAST_MODE=True</code>: حتى 300 شجرة وخيطَي CPU؛ ابدأ به.</li><li><code>FULL_MODE=True</code>: سقف اختياري يبلغ 900 شجرة بعد حفظ تجربة الأساس.</li><li>لا تحتاج إلى GPU أو مفتاح API أو ربط Drive أو اشتراك مدفوع.</li></ul>
<p><strong>شرط الاجتياز:</strong> ظهور <code>Day 1 setup verified</code>. لا تتجاوز فشل البصمة أو اختلاف إصدار مستورد؛ أعد تشغيل الجلسة ثم شغّل الدفتر من البداية.</p>
</td>
</tr>
</table>

In [1]:
from pathlib import Path
import hashlib, importlib.util, os, runpy, sys, time, urllib.request, urllib.error

RUN_STARTED = time.perf_counter()
COURSE_REVISION = "fe0c0204e6076a7ac2139b7336485a097343fb8a"
MANIFEST_SHA256 = "86d15e813caf833b7d86512808176388d4123dc3c01c6525ee849a76e8322c59"
SETUP_SHA256 = "79608f8f9559d9e6eb09d51ee541ee12d7e9262ac57d1ca3a948c8ecb14f0f35"
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
ROOT = Path("/content/tamweel") if IN_COLAB else Path(os.environ.get("TAMWEEL_PROJECT_DIR", ".")).resolve()
setup_path = ROOT / "scripts/setup_colab.py"
setup_path.parent.mkdir(parents=True, exist_ok=True)
if not setup_path.exists() or hashlib.sha256(setup_path.read_bytes()).hexdigest() != SETUP_SHA256:
    setup_url = f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{COURSE_REVISION}/scripts/setup_colab.py"
    for attempt in range(3):
        try:
            with urllib.request.urlopen(setup_url, timeout=45) as response:
                setup_bytes = response.read(200000)
            if hashlib.sha256(setup_bytes).hexdigest() != SETUP_SHA256:
                raise ValueError("Setup checksum mismatch. Reopen the released notebook.")
            setup_path.write_bytes(setup_bytes)
            break
        except (urllib.error.URLError, TimeoutError):
            if attempt == 2:
                raise RuntimeError("Download unavailable. Retry setup or upload the unchanged project files; see READINESS_GUIDE.")
            time.sleep(attempt + 1)
environment = runpy.run_path(str(setup_path))["prepare"](ROOT, COURSE_REVISION, MANIFEST_SHA256)
FAST_MODE, FULL_MODE, SEED, N_JOBS = True, False, 211, 2

from setup_colab import fetch_verified
LAB_REVISION = "c8344f6b19b26dcbce3dcfc671b746c59bf55895"
LAB_SHA256 = "2a968ba9e738cc5e3dae12077e576fdb40ec6ec64230613e445bbc7884e7e1ba"
fetch_verified(
    f"https://raw.githubusercontent.com/almiyead-rgb/sda-dsc-211-student-template/{LAB_REVISION}/scripts/day1_lab.py",
    ROOT / "scripts/day1_lab.py", LAB_SHA256)
print("Day 1 setup verified | إعداد اليوم الأول جاهز")

Package                 Version
numpy                   2.1.3
pandas                  2.2.3
scipy                   1.16.3
scikit-learn            1.6.1
matplotlib              3.10.0
xgboost                 3.4.1
lightgbm                4.6.0
shap                    0.52.0
numba                   0.61.2
optuna                  4.5.0
Python 3.12.13 | CPU | seed=211 | n_jobs=2 | FAST_MODE=True
Verified course files. Next: run the data and compatibility checks.


Day 1 setup verified | إعداد اليوم الأول جاهز


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>2. Understand the task and data</h2>
<p>All records are synthetic. The target <code>default_within_90d</code> records whether a default event occurs within 90 days after the application; it does not mean a delay of exactly 90 days.</p>
<ul><li>Use 22 features available at application time.</li><li>Exclude identifiers, dates, split-control columns and the target from model inputs.</li><li>Learn missing-value imputation from training rows only.</li><li>Do not use challenge data in this lab.</li></ul>

</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>2. افهم المهمة والبيانات</h2>
<p>جميع السجلات اصطناعية. يسجل الهدف <code>default_within_90d</code> ما إذا وقع حدث تعثر خلال 90 يومًا بعد الطلب؛ ولا يعني تأخرًا مدته 90 يومًا بالضبط.</p>
<ul><li>استخدم 22 خاصية متاحة وقت تقديم الطلب.</li><li>استبعد المعرّفات والتواريخ وأعمدة التحكم في التقسيم والهدف من مدخلات النموذج.</li><li>تعلّم تعويض القيم المفقودة من صفوف التدريب فقط.</li><li>لا تستخدم بيانات التحدي في هذا اللاب.</li></ul>

</td>
</tr>
</table>

In [2]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_curve, precision_recall_curve
from data_checks import load_course_data
from day1_lab import (LabConfig, MODEL_NAMES, make_splits, select_tree_count,
                      fit_model, compare_models, learner_checkpoint, config_dict)

warnings.filterwarnings("ignore", message="scipy.optimize:.*disp.*", category=DeprecationWarning)
frames, contract, data_summary = load_course_data(ROOT)
train = frames["train"]
FAST_MODE, FULL_MODE = True, False  # Choose exactly one mode.
assert FAST_MODE != FULL_MODE, "Choose exactly one mode."
config = LabConfig(max_trees=300 if FAST_MODE else 900)
X, y, splits = make_splits(train, contract, seed=config.seed)
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
display(pd.DataFrame({"rows": [len(train)], "predictors": [X.shape[1]],
                      "positive_rate": [y.mean()], "missing_cells": [int(X.isna().sum().sum())]}))
display(train[["application_id", "income_sar", "bureau_score", contract["target"]["name"]]].head(3))

Matplotlib is building the font cache; this may take a moment.


,rows,predictors,positive_rate,missing_cells
0,10000,22,0.0789,766


,application_id,income_sar,bureau_score,default_within_90d
0,TR-001121,8938.11,584.0,0
1,TR-002836,16995.65,652.0,0
2,TR-005579,8353.27,734.0,0


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>3. Separate data roles before fitting</h2>
<p>The 10,000 applications have distinct roles:</p>
<table><tr><th>Role</th><th>Rows</th><th>Purpose</th></tr><tr><td>Inner fit</td><td>6,000</td><td>Fit preprocessing and trees while selecting their count</td></tr><tr><td>Inner stop</td><td>2,000</td><td>Select the number of trees using log loss</td></tr><tr><td>Development</td><td>8,000</td><td>Refit each model after internal decisions are fixed</td></tr><tr><td>Comparison</td><td>2,000</td><td>Score all models once after fitting</td></tr></table>
<p>The inner sets are contained inside development; they are not extra data. The comparison set must not be used for early stopping or repeated tuning. This separation improves the teaching comparison, but customer and time leakage remain intentionally unresolved until Day 2.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>3. افصل أدوار البيانات قبل التدريب</h2>
<p>للطلبات البالغ عددها 10,000 أدوار منفصلة:</p>
<table><tr><th>الدور</th><th>الصفوف</th><th>الغرض</th></tr><tr><td>Inner fit</td><td>6,000</td><td>تعلم المعالجة والأشجار أثناء اختيار عددها</td></tr><tr><td>Inner stop</td><td>2,000</td><td>اختيار عدد الأشجار باستخدام log loss</td></tr><tr><td>Development</td><td>8,000</td><td>إعادة تدريب كل نموذج بعد تثبيت القرارات الداخلية</td></tr><tr><td>Comparison</td><td>2,000</td><td>تقييم جميع النماذج مرة واحدة بعد التدريب</td></tr></table>
<p>المجموعتان الداخليتان جزء من التطوير وليستا بيانات إضافية. لا تستخدم مجموعة المقارنة للإيقاف المبكر أو الضبط المتكرر. يحسن هذا الفصل المقارنة التعليمية، لكن تسرب العميل والزمن يبقيان دون معالجة عمدًا حتى اليوم الثاني.</p>
</td>
</tr>
</table>

In [3]:
# Keep the comparison rows out of training and early stopping.
development, comparison = splits["development"], splits["comparison"]
inner_fit, inner_stop = splits["inner_fit"], splits["inner_stop"]
assert not (set(development) & set(comparison))
assert not (set(inner_fit) & set(inner_stop))
split_table = pd.DataFrame([
    {"role": role, "rows": len(index), "positive_rate": y.loc[index].mean()}
    for role, index in splits.items()
])
shared_customers = len(set(train.loc[development, "customer_id"]) & set(train.loc[comparison, "customer_id"]))
display(split_table)
print(f"Customers shared across development/comparison: {shared_customers} — revisit on Day 2")
models, timings, selections = {}, {}, {}

,role,rows,positive_rate
0,development,8000,0.078875
1,comparison,2000,0.079000
2,inner_fit,6000,0.078833
3,inner_stop,2000,0.079000


Customers shared across development/comparison: 1226 — revisit on Day 2


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>4. Start with a simple baseline</h2>
<p>The baseline is a single Pipeline: median imputation, feature scaling and Logistic Regression. Every learned preprocessing value comes from development rows only.</p>
<p><strong>Why it matters:</strong> a more complex model must demonstrate value over a credible simple reference. Complexity is not evidence of improvement.</p>
<p><strong>Observe:</strong> training time, ROC-AUC and Average Precision. Do not reject the baseline merely because it does not use trees.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>4. ابدأ بخط أساس بسيط</h2>
<p>خط الأساس Pipeline واحدة تجمع التعويض بالوسيط وتوحيد المقاييس وLogistic Regression. تُتعلم جميع قيم المعالجة من صفوف التطوير فقط.</p>
<p><strong>لماذا يهم؟</strong> يجب أن يثبت النموذج الأكثر تعقيدًا قيمة تتجاوز مرجعًا بسيطًا وموثوقًا. التعقيد وحده ليس دليلًا على التحسن.</p>
<p><strong>راقب:</strong> زمن التدريب وROC-AUC وAverage Precision. لا تستبعد خط الأساس لمجرد أنه لا يستخدم الأشجار.</p>
</td>
</tr>
</table>

In [4]:
name = "Logistic Regression"
models[name], timings[name] = fit_model(name, X.loc[development], y.loc[development], config)
print(f"Baseline fitted on {len(development):,} rows in {timings[name]['train_seconds']:.3f} s")

Baseline fitted on 8,000 rows in 0.072 s


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>5. Compare boosting with early stopping</h2>
<p>Boosting adds trees sequentially to reduce the objective. Under logistic classification, tree contributions accumulate in log-odds space before conversion to probability; the model is not directly summing tree probabilities.</p>
<table><tr><th>Control</th><th>What to observe</th></tr><tr><td><code>learning_rate</code></td><td>Contribution of each tree; a smaller value may require more trees</td></tr><tr><td><code>max_trees</code></td><td>Search ceiling, not a mandatory tree count</td></tr><tr><td><code>xgb_depth</code> / <code>lgb_leaves</code></td><td>Tree complexity</td></tr><tr><td><code>min_child_samples</code></td><td>Prevents very small LightGBM leaves</td></tr><tr><td><code>patience</code></td><td>Rounds without improvement in inner-stop log loss</td></tr></table>
<p><code>selected_trees</code> is one-based. Logistic Regression has no tree count. Class weights are intentionally deferred to a later lab.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>5. قارن التعزيز باستخدام الإيقاف المبكر</h2>
<p>يضيف التعزيز أشجارًا بالتتابع لتقليل دالة الهدف. في التصنيف اللوجستي تتراكم إسهامات الأشجار في فضاء log-odds قبل تحويلها إلى احتمال؛ ولا يجمع النموذج احتمالات الأشجار مباشرة.</p>
<table><tr><th>الإعداد</th><th>ما الذي تراقبه؟</th></tr><tr><td><code>learning_rate</code></td><td>مساهمة كل شجرة؛ قد تتطلب القيمة الأصغر أشجارًا أكثر</td></tr><tr><td><code>max_trees</code></td><td>سقف البحث وليس عددًا إلزاميًا</td></tr><tr><td><code>xgb_depth</code> / <code>lgb_leaves</code></td><td>تعقيد الأشجار</td></tr><tr><td><code>min_child_samples</code></td><td>يمنع أوراق LightGBM الصغيرة جدًا</td></tr><tr><td><code>patience</code></td><td>عدد الجولات دون تحسن في inner-stop log loss</td></tr></table>
<p>يبدأ <code>selected_trees</code> من 1. لا يوجد عدد أشجار لـLogistic Regression. تؤجل أوزان الفئات عمدًا إلى لاب لاحق.</p>
</td>
</tr>
</table>

In [5]:
for name in ("XGBoost", "LightGBM"):
    selections[name] = select_tree_count(
        name, X.loc[inner_fit], y.loc[inner_fit], X.loc[inner_stop], y.loc[inner_stop], config)
    models[name], timings[name] = fit_model(
        name, X.loc[development], y.loc[development], config, selections[name])
    print(f"{name}: {selections[name]['selected_trees']} selected trees; "
          f"{selections[name]['rounds_evaluated']} evaluated rounds; "
          f"{timings[name]['train_seconds']:.3f} s including selection + refit")
    if selections[name]["hit_tree_budget"]:
        print("Tree budget reached: record this limitation before considering FULL_MODE.")

XGBoost: 82 selected trees; 112 evaluated rounds; 0.320 s including selection + refit


LightGBM: 41 selected trees; 71 evaluated rounds; 0.305 s including selection + refit


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Read the learning curves</h2>
<p>The dashed line shows inner-stop loss. The vertical marker shows the selected tree count. Falling training loss alone is not enough.</p>
<ul><li>Identify where inner-stop loss stops improving.</li><li>Record whether the search reached the tree budget.</li><li>Do not infer general superiority from one curve.</li></ul>
<p>Plot labels remain concise English technical labels so exported figures render consistently across environments.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>اقرأ منحنيات التعلم</h2>
<p>يمثل الخط المتقطع خسارة inner stop، وتحدد العلامة الرأسية عدد الأشجار المختار. انخفاض خسارة التدريب وحدها لا يكفي.</p>
<ul><li>حدد أين توقفت خسارة inner stop عن التحسن.</li><li>سجل ما إذا وصل البحث إلى سقف الأشجار.</li><li>لا تستنتج تفوقًا عامًا من منحنى واحد.</li></ul>
<p>تبقى عناوين الرسوم التقنية بالإنجليزية وبصياغة قصيرة لضمان ثبات عرض الملفات المصدّرة عبر البيئات المختلفة.</p>
</td>
</tr>
</table>

In [6]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
                     "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.2})
colors = {"Logistic Regression": "#2667A0", "XGBoost": "#BA5A17", "LightGBM": "#18816A"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, (name, result) in zip(axes, selections.items()):
    rounds = np.arange(1, len(result["curves"]["fit"]) + 1)
    ax.plot(rounds, result["curves"]["fit"], label="Inner fit (6,000)", color="#2667A0")
    ax.plot(rounds, result["curves"]["stop"], label="Inner stop (2,000)", color="#BA5A17", linestyle="--")
    ax.axvline(result["selected_trees"], color="#444444", linestyle=":",
               label=f"Selected: {result['selected_trees']} trees")
    ax.set(title=name, xlabel="Boosting round (1-based)", ylabel="Log loss (lower is better)")
    ax.legend(fontsize=9)
fig.suptitle("Tree count selected using inner rows only")
fig.savefig(ARTIFACTS / "day1_learning_curves.png", dpi=160)
plt.show()

/var/folders/mf/f8kr4l7d1rb3vy03cpjyz45r0000gp/T/ipykernel_12768/2033280558.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>6. Compare the three models fairly</h2>
<p><strong>ROC-AUC</strong> summarizes ranking of positive versus negative cases across thresholds. <strong>Average Precision (AP)</strong> summarizes precision–recall performance and is especially useful when the positive class is uncommon. In this course, PR-AUC refers to scikit-learn Average Precision, not trapezoidal area.</p>
<p><code>train_seconds</code> includes tree-count selection and refitting for boosting models, and fitting for Logistic Regression. Download and plotting time are excluded. Timing is an observation from this run, not a hardware guarantee.</p>
<p>This single split has no confidence interval and does not establish calibration, operational value or general superiority. Simulated decision cost <code>10 × FN + FP</code> and 12% review capacity are introduced in a later decision lab.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>6. قارن النماذج الثلاثة بعدالة</h2>
<p>يلخص <strong>ROC-AUC</strong> ترتيب الحالات الموجبة مقابل السالبة عبر العتبات. ويلخص <strong>Average Precision (AP)</strong> أداء precision–recall، ويكون مهمًا عند ندرة الفئة الموجبة. في هذه الدورة يشير PR-AUC إلى Average Precision في scikit-learn، وليس المساحة شبه المنحرفة.</p>
<p>يشمل <code>train_seconds</code> اختيار عدد الأشجار وإعادة التدريب لنماذج التعزيز، وتدريب Logistic Regression. لا يشمل زمن التنزيل والرسم. الزمن رصد لهذا التشغيل وليس ضمانًا على جهاز آخر.</p>
<p>لا تتضمن هذه التجربة الواحدة فترات ثقة، ولا تثبت المعايرة أو القيمة التشغيلية أو التفوق العام. ستُستخدم تكلفة القرار التعليمية <code>10 × FN + FP</code> وسعة مراجعة 12% في لاب قرار لاحق.</p>
</td>
</tr>
</table>

In [7]:
comparison_table, probabilities = compare_models(
    models, timings, selections, X.loc[comparison], y.loc[comparison])
comparison_table["selected_trees"] = comparison_table["selected_trees"].astype("Int64")
display(comparison_table[["model", "roc_auc", "average_precision", "train_seconds",
                          "selected_trees", "comparison_rows", "positive_rate"]].round(4))
comparison_table.to_csv(ARTIFACTS / "day1_model_comparison.csv", index=False)
prediction_export = train.loc[comparison, ["application_id", "default_within_90d"]].join(probabilities)
prediction_export.to_csv(ARTIFACTS / "day1_comparison_predictions.csv", index=False)
assert len(comparison_table) == 3
assert comparison_table[["roc_auc", "average_precision", "train_seconds"]].notna().all().all()
print("TECHNICAL_READY — three models compared; your interpretation is still required.")

,model,roc_auc,average_precision,train_seconds,selected_trees,comparison_rows,positive_rate
0,Logistic Regression,0.8213,0.3258,0.0719,<NA>,2000,0.079
1,XGBoost,0.8141,0.3416,0.3201,82,2000,0.079
2,LightGBM,0.8138,0.3248,0.3052,41,2000,0.079


TECHNICAL_READY — three models compared; your interpretation is still required.


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Read ROC and precision–recall together</h2>
<p>The ROC reference line represents random ranking. The precision reference line equals the positive prevalence in the comparison sample.</p>
<ul><li>Compare curve separation, not only one headline number.</li><li>Inspect precision when recall is high.</li><li>Return to the table for exact values and sample context.</li></ul>
<p>A visually higher curve on one random split is evidence for this run only.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>اقرأ ROC وprecision–recall معًا</h2>
<p>يمثل الخط المرجعي في ROC ترتيبًا عشوائيًا، بينما يساوي خط precision المرجعي نسبة الفئة الموجبة في عينة المقارنة.</p>
<ul><li>قارن تباعد المنحنيات، ولا تعتمد على رقم رئيس واحد فقط.</li><li>افحص precision عندما يكون recall مرتفعًا.</li><li>ارجع إلى الجدول للقيم الدقيقة وسياق حجم العينة.</li></ul>
<p>ارتفاع منحنى بصريًا على تقسيم عشوائي واحد دليل يخص هذا التشغيل فقط.</p>
</td>
</tr>
</table>

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
styles = ["-", "--", "-."]
for name, style in zip(MODEL_NAMES, styles):
    fpr, tpr, _ = roc_curve(y.loc[comparison], probabilities[name])
    precision, recall, _ = precision_recall_curve(y.loc[comparison], probabilities[name])
    row = comparison_table.set_index("model").loc[name]
    axes[0].plot(fpr, tpr, color=colors[name], linestyle=style, label=f"{name}: {row.roc_auc:.3f}")
    axes[1].plot(recall, precision, color=colors[name], linestyle=style, label=f"{name}: AP {row.average_precision:.3f}")
axes[0].plot([0, 1], [0, 1], color="#777777", linestyle=":", label="Random ranking")
axes[1].axhline(y.loc[comparison].mean(), color="#777777", linestyle=":",
               label=f"Prevalence: {y.loc[comparison].mean():.1%}")
axes[0].set(title="ROC", xlabel="False positive rate", ylabel="True positive rate")
axes[1].set(title="Precision–recall", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
axes[0].legend(fontsize=8, loc="lower right")
axes[1].legend(fontsize=8, loc="upper right")
fig.suptitle(f"Teaching comparison · {len(comparison):,} synthetic requests · random split")
fig.savefig(ARTIFACTS / "day1_roc_pr.png", dpi=160)
plt.show()

/var/folders/mf/f8kr4l7d1rb3vy03cpjyz45r0000gp/T/ipykernel_12768/2256449554.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>7. Own the model-selection decision</h2>
<p>Complete the next shared code cell in your own words. There is no automatic winner and no model is required to win.</p>
<ol><li><strong>Problem statement:</strong> outcome, decision time and intended teaching use.</li><li><strong>Candidate:</strong> choose one of the three models.</li><li><strong>Evidence:</strong> cite a number or comparison from your run.</li><li><strong>Limitation:</strong> state what this experiment cannot prove.</li><li><strong>Next test:</strong> identify evidence that could change your choice.</li></ol>
<p><strong>Exit questions:</strong> Why inspect AP beside ROC-AUC under class imbalance? Why must the comparison set remain separate from early stopping?</p>
<p>The automated checkpoint detects empty fields only. It does not judge correctness, award a grade or prove authorship.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>7. امتلك قرار اختيار النموذج</h2>
<p>أكمل خلية الكود المشتركة التالية بصياغتك أنت. لا يوجد فائز تلقائي، ولا يُشترط أن يفوز نموذج محدد.</p>
<ol><li><strong>صياغة المشكلة:</strong> النتيجة ولحظة القرار والاستخدام التعليمي.</li><li><strong>المرشح:</strong> اختر أحد النماذج الثلاثة.</li><li><strong>الدليل:</strong> استشهد برقم أو مقارنة من تشغيلك.</li><li><strong>القيد:</strong> وضح ما لا تستطيع التجربة إثباته.</li><li><strong>الاختبار التالي:</strong> حدد دليلًا قد يغير اختيارك.</li></ol>
<p><strong>سؤالا الخروج:</strong> لماذا تفحص AP بجانب ROC-AUC عند عدم توازن الفئات؟ ولماذا يجب فصل مجموعة المقارنة عن الإيقاف المبكر؟</p>
<p>يكتشف الفحص الآلي الخانات الفارغة فقط؛ لا يحكم على صحة الإجابة ولا يمنح درجة ولا يثبت الملكية الفكرية.</p>
</td>
</tr>
</table>

In [9]:
problem_statement = ('Estimate the risk of synthetic default within 90 days using information available when a request '
 'arrives. The score supports a simulated manual review queue.')
candidate = 'XGBoost'
notes = {'evidence': 'XGBoost led on Average Precision at 0.3338 versus 0.3258 for Logistic Regression. '
             'Logistic Regression led on ROC-AUC at 0.8213.',
 'limitation': 'The comparison was a single random split of operations, with 1,226 customers appearing '
               'on both sides. The AP difference was small and does not establish future performance.',
 'next_test': 'Compare on later applications with customer separation and a 90-day label-maturity gap. '
              'Check whether the AP advantage persists across periods.'}
exit_answers = ['Only 158 of 2,000 applications defaulted. AP focuses on the precision and recall of that rare class, '
 'which is the workload the review team cares about. ROC-AUC also captures how the score ranks classes '
 'overall.',
 'Early stopping chooses a tree count using observed performance. The inner stopping split serves that '
 'purpose. Keeping the 2,000 comparison rows out of this choice preserves a separate estimate.']
checkpoint = learner_checkpoint(candidate, notes, problem_statement, exit_answers)
print(checkpoint["status"])
for reminder in checkpoint["missing"]:
    print("-", reminder)

READY_FOR_REVIEW


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>8. Export evidence, not screenshots</h2>
<p>The export cell writes eight required files and creates <code>day1_artifacts.zip</code>. Download the ZIP, extract it and upload the actual files to <code>artifacts/</code>. Save the executed notebook in <code>notebooks/</code>.</p>
<ul><li><code>TECHNICAL_READY</code>: the models and evidence were produced.</li><li><code>LEARNER_WORK_REQUIRED</code>: technical execution passed, but your written fields remain incomplete.</li><li><code>READY_FOR_REVIEW</code>: required fields are present; this is not a grade or correctness guarantee.</li></ul>
<p>Copy your problem statement into your project README. Never upload access tokens, passwords or personal data.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>8. صدّر الأدلة وليس لقطات الشاشة فقط</h2>
<p>تكتب خلية التصدير ثمانية ملفات إلزامية وتنشئ <code>day1_artifacts.zip</code>. نزّل الحزمة وافكها وارفع الملفات الفعلية إلى <code>artifacts/</code>. واحفظ الدفتر المنفذ داخل <code>notebooks/</code>.</p>
<ul><li><code>TECHNICAL_READY</code>: تم إنتاج النماذج والأدلة.</li><li><code>LEARNER_WORK_REQUIRED</code>: نجح التنفيذ التقني، لكن حقولك الكتابية غير مكتملة.</li><li><code>READY_FOR_REVIEW</code>: الحقول المطلوبة موجودة؛ ولا تمثل درجة أو ضمانًا لصحة التفسير.</li></ul>
<p>انسخ صياغة المشكلة إلى README الخاص بمشروعك. لا ترفع رموز الدخول أو كلمات المرور أو البيانات الشخصية.</p>
</td>
</tr>
</table>

In [10]:
import platform, zipfile

checkpoint = learner_checkpoint(candidate, notes, problem_statement, exit_answers)
reflection = {"problem_statement": problem_statement, "candidate": candidate,
              "notes": notes, "exit_answers": exit_answers, "checkpoint": checkpoint}
(ARTIFACTS / "day1_reflection.json").write_text(json.dumps(reflection, ensure_ascii=False, indent=2), encoding="utf-8")
membership = train[["application_id"]].copy()
membership["role"] = "comparison"
membership.loc[inner_fit, "role"] = "inner_fit"
membership.loc[inner_stop, "role"] = "inner_stop"
membership.to_csv(ARTIFACTS / "day1_split_membership.csv", index=False)
report = {"technical_status": "TECHNICAL_READY", "learner_status": checkpoint["status"],
          "data_revision": COURSE_REVISION, "lab_revision": LAB_REVISION,
          "lab_sha256": LAB_SHA256, "data_manifest_sha256": MANIFEST_SHA256,
          "training_file_sha256": hashlib.sha256((ROOT / "data/tamweel_train.csv").read_bytes()).hexdigest(),
          "config": config_dict(config), "mode": "FAST" if FAST_MODE else "FULL",
          "python": platform.python_version(), "split": split_table.to_dict("records"),
          "split_method": "random stratified teaching baseline; not final validation",
          "shared_customers": shared_customers, "selections": selections,
          "ap_definition": "sklearn average_precision_score, non-interpolated",
          "elapsed_seconds": round(time.perf_counter() - RUN_STARTED, 2)}
if IN_COLAB:
    import resource
    report["peak_process_memory_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024, 1)
(ARTIFACTS / "day1_run.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
export_names = ["environment.json", "day1_model_comparison.csv", "day1_comparison_predictions.csv",
                "day1_split_membership.csv", "day1_learning_curves.png", "day1_roc_pr.png",
                "day1_reflection.json", "day1_run.json"]
with zipfile.ZipFile(ARTIFACTS / "day1_artifacts.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in export_names:
        bundle.write(ARTIFACTS / filename, filename)
print(f"TECHNICAL_READY | {checkpoint['status']} | {report['elapsed_seconds']} seconds | CPU")
print("Saved day1_artifacts.zip — download it and save your notebook.")

TECHNICAL_READY | READY_FOR_REVIEW | 35.56 seconds | CPU
Saved day1_artifacts.zip — download it and save your notebook.


<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Completion checkpoint and next step</h2>
<ul><li>Three valid model rows appear in <code>day1_model_comparison.csv</code>.</li><li>You can explain development, inner stopping and comparison roles.</li><li>You selected one candidate and documented evidence, a limitation and a next test.</li><li>You saved the executed notebook and all eight artifacts outside the temporary Colab session.</li></ul>
<p><strong>Optional extension:</strong> after preserving the baseline, change <code>learning_rate</code> once with the same split. Document the impact on tree count and time. Repeatedly searching for the highest comparison score would convert the comparison set into a tuning set.</p>
<p><strong>Troubleshooting:</strong> retry setup after a download interruption; restart the session after an imported-version mismatch; restore original files after a checksum failure; never replace your outputs with another learner's table.</p>
<p><strong>Day 2:</strong> test customer separation, time ordering and target maturity, then reassess whether the first candidate remains credible.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>نقطة الإكمال والخطوة التالية</h2>
<ul><li>تظهر ثلاثة صفوف صالحة في <code>day1_model_comparison.csv</code>.</li><li>تستطيع شرح أدوار التطوير والإيقاف الداخلي والمقارنة.</li><li>اخترت مرشحًا ووثقت دليلًا وقيدًا واختبارًا تاليًا.</li><li>حفظت الدفتر المنفذ والملفات الثمانية خارج جلسة Colab المؤقتة.</li></ul>
<p><strong>توسع اختياري:</strong> بعد حفظ تجربة الأساس، غيّر <code>learning_rate</code> مرة واحدة مع التقسيم نفسه، ووثق أثره على عدد الأشجار والزمن. البحث المتكرر عن أعلى نتيجة يحول مجموعة المقارنة إلى مجموعة ضبط.</p>
<p><strong>حل التعثر:</strong> أعد خلية الإعداد عند انقطاع التنزيل؛ أعد تشغيل الجلسة عند اختلاف إصدار مستورد؛ استرجع الملفات الأصلية عند فشل البصمة؛ ولا تستبدل مخرجاتك بجدول متدرب آخر.</p>
<p><strong>اليوم الثاني:</strong> اختبر فصل العملاء وترتيب الزمن ونضج الهدف، ثم أعد تقييم مدى موثوقية المرشح الأول.</p>
</td>
</tr>
</table>

<table style="width:100%; border-collapse:collapse;">
<tr>
<td width="50%" valign="top" dir="ltr" style="padding:14px; border:1px solid #d0d7de;">
<h2>Daily question · Model comparison</h2>
<p><strong>Which model did you select initially, and what is the strongest result supporting your choice?</strong><br><br>Write a short answer under <code>Day 1</code> in the <code>Daily answers | الإجابات اليومية</code> section of your repository README. Then download this executed notebook and the required generated files and upload them on the same day.</p>
</td>
<td width="50%" valign="top" dir="rtl" style="padding:14px; border:1px solid #d0d7de;">
<h2>سؤال اليوم · مقارنة النماذج</h2>
<p><strong>أي نموذج اخترت مبدئيًا، وما الدليل الأقوى من نتائجك؟</strong><br><br>اكتب إجابة مختصرة تحت <code>Day 1 | اليوم 1</code> في قسم <code>Daily answers | الإجابات اليومية</code> داخل README في مستودعك، ثم نزّل هذا الدفتر بعد التنفيذ وملفات النتائج المطلوبة وارفعها في اليوم نفسه.</p>
</td>
</tr>
</table>

## Daily answer | إجابة اليوم

I picked XGBoost for now. Only 7.9% of the requests in the comparison set defaulted, so I paid more attention to how well each model ranked those cases. XGBoost had the highest Average Precision at 0.3338, just ahead of Logistic Regression at 0.3258. The gap was small, so I’ll see if it holds up on Day 2.

**Why I changed the validation.** The Day 1 split separated application rows, but 1,226 customers still appeared in both development and comparison. Day 2 addressed that overlap and time ordering using LightGBM as the course lab’s common model. The comparison across Day 1’s three models still stands as a baseline for that original split.